[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/optimization/01_problem_formulation_and_convexity/exercises.ipynb)

# Module 01 Exercises — Optimization Problem Formulation and Convexity

A single preamble sets up NumPy, SciPy and the random generator used by every
verification cell below. Each problem's boxed answer is recomputed in code immediately after the
solution, so no number in this notebook rests on memory.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linprog

plt.rcParams.update({
    "figure.figsize": (7.0, 4.0),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

print("environment ready:", np.__version__)

environment ready: 2.4.6


## L0 — Concept Checks

### Problem L0.1 — From Maximization to Standard Form

**Problem Statement:** Rewrite the problem "maximize $3x + 2y$ subject to $x + y \le 4$, $x \ge 0$, $y \ge 1$" in the standard form $\min f$ s.t. $g_i \le 0$, and identify the objective, the constraints, and the decision variable.

*Intuition:* Standard form is a bookkeeping convention: flip the objective's sign for maximization and move every constraint to the form "something $\le 0$".

**Solution:**

**Step 1 (Objective).** Maximizing $3x + 2y$ is equivalent to minimizing its negative, so set $f(x, y) = -3x - 2y$.

**Step 2 (Constraints to $\le 0$ form).**

- $x + y \le 4$ becomes $g_1(x, y) = x + y - 4 \le 0$.
- $x \ge 0$ becomes $g_2(x, y) = -x \le 0$.
- $y \ge 1$ becomes $g_3(x, y) = 1 - y \le 0$.

There are no equality constraints ($p = 0$). The decision variable is $\mathbf{x} = (x, y) \in \mathbb{R}^2$.

**Step 3 (Assembled standard form).**

$$
\boxed{\min_{(x,y) \in \mathbb{R}^2} \ -3x - 2y \quad \text{s.t.} \quad x + y - 4 \le 0, \quad -x \le 0, \quad 1 - y \le 0}
$$

If $(x^{\star}, y^{\star})$ solves this problem with optimal value $p^{\star}$, the original maximum equals $-p^{\star}$ at the same point.

> **Key takeaway:** Any mix of max/min objectives and $\ge$, $\le$, $=$ constraints can be normalized mechanically; theory and solvers only ever need the one standard form.

In [2]:
# Problem L0.1 - solve the standard-form LP and confirm max f = -p*
res = linprog(c=[-3.0, -2.0], A_ub=[[1.0, 1.0]], b_ub=[4.0],
              bounds=[(0, None), (1, None)])
print(f"standard-form optimal value p*  = {res.fun:.4f}")
print(f"minimizer (x, y)                = {res.x}")
print(f"original maximum = -p*          = {-res.fun:.4f}")
assert abs(res.fun + 11.0) < 1e-9 and np.allclose(res.x, [3.0, 1.0])

standard-form optimal value p*  = -11.0000
minimizer (x, y)                = [3. 1.]
original maximum = -p*          = 11.0000


### Problem L0.2 — Feasible Set, Optimal Value, and Minimizer

**Problem Statement:** For the problem $\min x_1$ subject to $x_1 + x_2 = 1$, $x_1 \ge 0$, $x_2 \ge 0$, describe the feasible set $\mathcal{F}$ geometrically, compute the optimal value $p^{\star}$, and find all minimizers.

*Intuition:* The feasible set is a line segment in the plane; minimizing a coordinate over it means walking to one end.

**Solution:**

**Step 1 (Describe $\mathcal{F}$).** The equality constraint places $\mathbf{x}$ on the line $x_1 + x_2 = 1$; the two inequalities cut it down to the segment joining $(1, 0)$ and $(0, 1)$:

$$
\mathcal{F} = \{(x_1, x_2) \mid x_1 + x_2 = 1, \ x_1 \ge 0, \ x_2 \ge 0\} = \{(t, 1-t) \mid t \in [0, 1]\}
$$

This is a bounded polyhedron (a simplex), hence convex and compact.

**Step 2 (Optimize).** On $\mathcal{F}$ the objective is $f(t, 1-t) = t$ with $t \in [0, 1]$, minimized at $t = 0$.

**Step 3 (Conclusion).** The optimal value and the unique minimizer are

$$
\boxed{p^{\star} = 0, \qquad \mathbf{x}^{\star} = (0, 1)}
$$

Existence was automatic here: a continuous objective over a nonempty compact set attains its minimum (Weierstrass).

> **Key takeaway:** Formulating means exhibiting three objects — the feasible set, the optimal value, and the (possibly empty) set of minimizers — and each must be identified separately.

In [3]:
# Problem L0.2 - optimal value and minimizer of min x1 on the unit simplex
res = linprog(c=[1.0, 0.0], A_eq=[[1.0, 1.0]], b_eq=[1.0],
              bounds=[(0, None), (0, None)])
print(f"p*  = {res.fun:.6f}    (hand: 0)")
print(f"x*  = {res.x}    (hand: (0, 1))")
assert abs(res.fun) < 1e-12 and np.allclose(res.x, [0.0, 1.0], atol=1e-9)

p*  = 0.000000    (hand: 0)
x*  = [0. 1.]    (hand: (0, 1))


### Problem L0.3 — Local versus Global Minima — True or False

**Problem Statement:** Decide, with justification or counterexample: (a) "Every global minimizer is a local minimizer." (b) "Every local minimizer is a global minimizer." (c) "For a convex problem, every local minimizer is global."

*Intuition:* Global beats everyone, local only beats the neighbors; convexity is exactly the property that closes the gap.

**Solution:**

**(a) True.** If $f(\mathbf{x}^{\star}) \le f(\mathbf{x})$ for *all* feasible $\mathbf{x}$, the inequality holds in particular for feasible points within any $\varepsilon$-ball of $\mathbf{x}^{\star}$, which is the definition of a local minimizer.

**(b) False.** Take $f(x) = x^4 - 2x^2 + x$ on $\mathbb{R}$. Then $f'(x) = 4x^3 - 4x + 1$, whose three real roots are $x \approx -1.1072,\ 0.2696,\ 0.8376$. Evaluating $f$ at the two roots that are local minimizers gives $f(-1.1072) \approx -2.056$ and $f(0.8376) \approx -0.073$: two unequal basins, so $x \approx 0.8376$ is a strict local minimizer that is not global.

**(c) True.** This is Theorem 4.6 of [`first_principles.ipynb`](first_principles.ipynb): if $f$ and $\mathcal{F}$ are convex and $\mathbf{y}$ beat a local minimizer $\mathbf{x}^{\star}$, then points $(1-\theta)\mathbf{x}^{\star} + \theta\mathbf{y}$ with small $\theta \gt 0$ are feasible, arbitrarily close to $\mathbf{x}^{\star}$, and have value strictly below $f(\mathbf{x}^{\star})$ — contradiction.

$$
\boxed{\text{(a) true} \qquad \text{(b) false (double-well counterexample)} \qquad \text{(c) true}}
$$

> **Key takeaway:** "Local implies global" is not a fact about minimization in general — it is the defining privilege of convex problems.

In [4]:
# Problem L0.3 - the double-well counterexample, with real numbers
f03 = lambda x: x**4 - 2.0 * x**2 + x
crit = np.sort(np.roots([4.0, 0.0, -4.0, 1.0]).real)
print("critical points of f'(x) = 4x^3 - 4x + 1 :", crit)
print("objective values there                   :", f03(crit))

loc, glob = crit[2], crit[0]
window = np.linspace(loc - 0.2, loc + 0.2, 2001)
print(f"\nx = {loc:.4f} beats its whole neighbourhood : {f03(loc) <= f03(window).min() + 1e-12}")
print(f"but f({glob:.4f}) = {f03(glob):.4f} < f({loc:.4f}) = {f03(loc):.4f}, so it is not global")
assert f03(loc) <= f03(window).min() + 1e-12 and f03(glob) < f03(loc)

critical points of f'(x) = 4x^3 - 4x + 1 : [-1.1072  0.2696  0.8376]
objective values there                   : [-2.0562  0.1295 -0.0733]

x = 0.8376 beats its whole neighbourhood : True
but f(-1.1072) = -2.0562 < f(0.8376) = -0.0733, so it is not global


### Problem L0.4 — Convexity of $x^2$ from the Bare Definition

**Problem Statement:** Using only the definition of a convex function (no derivatives), prove that $f(x) = x^2$ is convex on $\mathbb{R}$, and determine when the defining inequality is strict.

*Intuition:* The chord over a parabola visibly lies above it; algebra should confirm the picture with an exact nonnegative discrepancy.

**Solution:**

**Step 1 (Set up the two sides).** Fix $x, y \in \mathbb{R}$ and $\theta \in [0, 1]$. We must compare $f(\theta x + (1-\theta)y) = (\theta x + (1-\theta)y)^2$ against $\theta x^2 + (1-\theta)y^2$.

**Step 2 (Compute the gap exactly).**

$$
\theta x^2 + (1-\theta)y^2 - (\theta x + (1-\theta)y)^2 = \theta(1-\theta)x^2 - 2\theta(1-\theta)xy + \theta(1-\theta)y^2
$$

using $\theta - \theta^2 = \theta(1-\theta)$ and $(1-\theta) - (1-\theta)^2 = \theta(1-\theta)$. Factoring,

$$
\theta x^2 + (1-\theta)y^2 - (\theta x + (1-\theta)y)^2 = \theta(1-\theta)(x - y)^2
$$

**Step 3 (Conclude).** The right side is a product of nonnegative factors, so it is $\ge 0$, proving the chord inequality. It is *strictly* positive exactly when $\theta \in (0,1)$ and $x \neq y$, so $f$ is in fact strictly convex.

$$
\boxed{(\theta x + (1-\theta)y)^2 \le \theta x^2 + (1-\theta)y^2 \ \text{ with equality iff } \theta \in \{0,1\} \text{ or } x = y}
$$

> **Key takeaway:** Definition-level proofs of convexity reduce to exhibiting the chord-minus-function gap as an explicitly nonnegative quantity — here $\theta(1-\theta)(x-y)^2$.

In [5]:
# Problem L0.4 - the chord gap of x^2 equals theta(1-theta)(x-y)^2 exactly
xs = rng.uniform(-5, 5, 5000)
ys = rng.uniform(-5, 5, 5000)
th = rng.uniform(0, 1, 5000)
gap = th * xs**2 + (1 - th) * ys**2 - (th * xs + (1 - th) * ys) ** 2
formula = th * (1 - th) * (xs - ys) ** 2
print(f"max |gap - theta(1-theta)(x-y)^2| over 5000 triples : {np.max(np.abs(gap - formula)):.3e}")
print(f"minimum gap observed                                : {gap.min():.3e}  (>= 0)")
assert np.max(np.abs(gap - formula)) < 1e-10 and gap.min() >= -1e-12

max |gap - theta(1-theta)(x-y)^2| over 5000 triples : 9.603e-15
minimum gap observed                                : 2.407e-08  (>= 0)


## L1 — Foundations

### Problem L1.1 — Absolute Values via the Epigraph Trick

**Problem Statement:** Show that the non-smooth problem $\min_{x \in \mathbb{R}} \lvert x - 1\rvert + \lvert x + 2\rvert$ is equivalent to a linear program by introducing epigraph variables, and solve it.

*Intuition:* $\lvert u\rvert$ is the smallest $t$ satisfying $u \le t$ and $-u \le t$; lifting each absolute value into its epigraph linearizes the problem.

**Solution:**

**Step 1 (Lift each term).** Introduce $t_1, t_2$ and note $\lvert x-1\rvert \le t_1 \iff x - 1 \le t_1$ and $-(x-1) \le t_1$; similarly for $t_2$. Since we are minimizing $t_1 + t_2$, the constraints will be tight at the optimum ($t_i = \lvert \cdot \rvert$), so

$$
\min_{x, t_1, t_2} \ t_1 + t_2 \quad \text{s.t.} \quad x - 1 \le t_1, \ 1 - x \le t_1, \ x + 2 \le t_2, \ -x - 2 \le t_2
$$

is an equivalent problem with linear objective and four linear inequality constraints — a linear program.

**Step 2 (Verify equivalence).** For fixed $x$, the smallest feasible $t_1$ is $\max(x-1, 1-x) = \lvert x-1\rvert$ and the smallest $t_2$ is $\lvert x+2\rvert$; substituting these optimal values recovers the original objective, so both problems share the same optimal value and the same optimal $x$.

**Step 3 (Solve).** The function $\varphi(x) = \lvert x-1\rvert + \lvert x+2\rvert$ is the sum of distances from $x$ to $1$ and to $-2$. For any point between the two anchors the sum equals the distance between them, namely $3$; outside the interval it exceeds $3$. Hence

$$
\boxed{p^{\star} = 3, \qquad \operatorname{argmin} = [-2,\, 1]}
$$

The minimizer set is an interval — convex, as guaranteed for convex problems, but not a single point since $\varphi$ is not strictly convex.

> **Key takeaway:** Epigraph variables convert piecewise-linear convex objectives into linear programs; non-smoothness is a formulation artifact, not an intrinsic obstacle.

In [6]:
# Problem L1.1 - the value 3 and the argmin interval [-2, 1], by grid and by LP
phi = lambda x: np.abs(x - 1.0) + np.abs(x + 2.0)
grid = np.linspace(-5, 5, 100_001)
vals = phi(grid)
opt = grid[vals <= vals.min() + 1e-12]
print(f"grid minimum value : {vals.min():.6f}   (hand: 3)")
print(f"grid argmin        : [{opt.min():.4f}, {opt.max():.4f}]   (hand: [-2, 1])")

lp = linprog(c=[0.0, 1.0, 1.0],
             A_ub=[[1, -1, 0], [-1, -1, 0], [1, 0, -1], [-1, 0, -1]],
             b_ub=[1.0, -1.0, -2.0, 2.0],
             bounds=[(None, None)] * 3)
print(f"LP form optimal value : {lp.fun:.6f}   (same problem, linear constraints)")
assert abs(vals.min() - 3.0) < 1e-9 and abs(lp.fun - 3.0) < 1e-7

grid minimum value : 3.000000   (hand: 3)
grid argmin        : [-2.0000, 1.0000]   (hand: [-2, 1])
LP form optimal value : 3.000000   (same problem, linear constraints)


### Problem L1.2 — Half-Spaces and Polyhedra Are Convex

**Problem Statement:** Prove that the half-space $H = \{\mathbf{x} \in \mathbb{R}^n \mid \mathbf{a}^\top \mathbf{x} \le b\}$ (with $\mathbf{a} \neq \mathbf{0}$) is convex, and deduce that every polyhedron $P = \{\mathbf{x} \mid A\mathbf{x} \le \mathbf{b}\}$ is convex.

*Intuition:* A linear function evaluated on a segment interpolates its endpoint values, so it cannot exceed a bound satisfied at both ends.

**Solution:**

**Step 1 (Half-space).** Let $\mathbf{x}, \mathbf{y} \in H$ and $\theta \in [0,1]$. By linearity of the inner product,

$$
\mathbf{a}^\top \big(\theta\mathbf{x} + (1-\theta)\mathbf{y}\big) = \theta\,\mathbf{a}^\top \mathbf{x} + (1-\theta)\,\mathbf{a}^\top \mathbf{y} \le \theta b + (1-\theta) b = b
$$

where the inequality uses $\mathbf{a}^\top \mathbf{x} \le b$, $\mathbf{a}^\top \mathbf{y} \le b$ and the nonnegativity of $\theta$ and $1-\theta$. Hence the combination lies in $H$, and $H$ is convex. (The same computation with equality shows hyperplanes are convex.)

**Step 2 (Polyhedron as intersection).** Writing the rows of $A$ as $\mathbf{a}_i^\top$, the polyhedron is

$$
P = \bigcap_{i=1}^{m} \{\mathbf{x} \mid \mathbf{a}_i^\top \mathbf{x} \le b_i\}
$$

an intersection of finitely many half-spaces.

**Step 3 (Apply the intersection theorem).** An arbitrary intersection of convex sets is convex: if $\mathbf{x}, \mathbf{y}$ lie in every $C_\alpha$, then any convex combination lies in every $C_\alpha$, hence in the intersection. Applying this to Step 2:

$$
\boxed{\text{every half-space, hyperplane, and polyhedron } \{\mathbf{x} \mid A\mathbf{x} \le \mathbf{b}\} \text{ is convex}}
$$

> **Key takeaway:** Linear constraints are the atoms of convex feasibility; stacking any number of them can never create non-convexity.

In [7]:
# Problem L1.2 - no convex combination of feasible points ever leaves a polyhedron
A_poly = rng.normal(size=(5, 3))
b_poly = rng.uniform(1.0, 3.0, size=5)
cand = rng.uniform(-1.0, 1.0, size=(20_000, 3))
inside = cand[np.all(cand @ A_poly.T <= b_poly, axis=1)][:400]
print(f"feasible sample points collected : {len(inside)}")
assert len(inside) >= 200

i = rng.integers(0, len(inside), 2000)
j = rng.integers(0, len(inside), 2000)
t = rng.uniform(0, 1, 2000)[:, None]
combos = t * inside[i] + (1 - t) * inside[j]
escapes = int(np.sum(np.any(combos @ A_poly.T > b_poly + 1e-12, axis=1)))
print(f"convex combinations leaving the polyhedron : {escapes} of 2000")
assert escapes == 0

feasible sample points collected : 400
convex combinations leaving the polyhedron : 0 of 2000


### Problem L1.3 — Norm Balls Are Convex

**Problem Statement:** Let $\lVert \cdot\rVert$ be any norm on $\mathbb{R}^n$. Prove that the ball $B = \{\mathbf{x} \mid \lVert \mathbf{x} - \mathbf{c}\rVert \le r\}$ is convex.

*Intuition:* The triangle inequality is precisely the statement that mixing two short vectors cannot produce a long one.

**Solution:**

**Step 1 (Set up).** Let $\mathbf{x}, \mathbf{y} \in B$, $\theta \in [0,1]$, and write the combination's deviation from the center as

$$
\theta\mathbf{x} + (1-\theta)\mathbf{y} - \mathbf{c} = \theta(\mathbf{x} - \mathbf{c}) + (1-\theta)(\mathbf{y} - \mathbf{c})
$$

using $\theta\mathbf{c} + (1-\theta)\mathbf{c} = \mathbf{c}$.

**Step 2 (Triangle inequality and homogeneity).** Applying the two defining norm axioms,

$$
\lVert \theta(\mathbf{x}-\mathbf{c}) + (1-\theta)(\mathbf{y}-\mathbf{c})\rVert \le \lVert\theta(\mathbf{x}-\mathbf{c})\rVert + \lVert(1-\theta)(\mathbf{y}-\mathbf{c})\rVert = \theta\lVert\mathbf{x}-\mathbf{c}\rVert + (1-\theta)\lVert\mathbf{y}-\mathbf{c}\rVert
$$

where absolute homogeneity used $\theta, 1-\theta \ge 0$.

**Step 3 (Bound by $r$).** Since $\lVert\mathbf{x}-\mathbf{c}\rVert \le r$ and $\lVert\mathbf{y}-\mathbf{c}\rVert \le r$,

$$
\lVert\theta\mathbf{x} + (1-\theta)\mathbf{y} - \mathbf{c}\rVert \le \theta r + (1-\theta) r = r
$$

so the combination lies in $B$.

$$
\boxed{\text{norm balls are convex for every norm } (\ell_1, \ell_2, \ell_\infty, \dots)}
$$

The same argument shows $f(\mathbf{x}) = \lVert\mathbf{x}\rVert$ is itself a convex *function*, since Step 2 is exactly its chord inequality.

> **Key takeaway:** Convexity of balls (and of norms as functions) is a repackaging of the triangle inequality — no geometry of the particular norm is needed.

In [8]:
# Problem L1.3 - convexity of norm balls, for the l1, l2 and l-infinity norms
centre, radius = np.array([0.5, -1.0]), 2.0
for name, p in [("l1", 1), ("l2", 2), ("l-inf", np.inf)]:
    cand = rng.uniform(-4, 4, size=(20_000, 2))
    ball = cand[np.linalg.norm(cand - centre, p, axis=1) <= radius][:500]
    t = rng.uniform(0, 1, len(ball))[:, None]
    mixed = t * ball + (1 - t) * ball[rng.permutation(len(ball))]
    worst = np.linalg.norm(mixed - centre, p, axis=1).max()
    print(f"  {name:6s} ball: {len(ball):4d} points, worst combined radius = {worst:.6f} <= {radius}")
    assert worst <= radius + 1e-12

  l1     ball:  500 points, worst combined radius = 1.969109 <= 2.0
  l2     ball:  500 points, worst combined radius = 1.976460 <= 2.0
  l-inf  ball:  500 points, worst combined radius = 1.975720 <= 2.0


### Problem L1.4 — Strictly but Not Strongly Convex

**Problem Statement:** Show that $f(x) = x^4$ is strictly convex on $\mathbb{R}$ but not $\mu$-strongly convex for any $\mu \gt 0$.

*Intuition:* Near the origin the quartic is flatter than every parabola, so it cannot dominate one; away from flat spots it still bends strictly upward.

**Solution:**

**Step 1 (Strict convexity).** $f'(x) = 4x^3$ is *strictly increasing* on $\mathbb{R}$. For $x \lt y$ and $z = \theta x + (1-\theta)y$ with $\theta \in (0,1)$, the mean value theorem gives points $\xi_1 \in (x, z)$ and $\xi_2 \in (z, y)$ with

$$
\frac{f(z) - f(x)}{z - x} = f'(\xi_1) \lt f'(\xi_2) = \frac{f(y) - f(z)}{y - z}
$$

Cross-multiplying the strict slope inequality (all denominators positive) and rearranging with $z - x = (1-\theta)(y-x)$ and $y - z = \theta(y-x)$ yields $f(z) \lt \theta f(x) + (1-\theta)f(y)$: strict convexity. (Note $f'' = 12x^2$ vanishes at one point, so the naive test $f'' \gt 0$ everywhere is sufficient but not necessary.)

**Step 2 (Failure of strong convexity).** By definition, $f$ is $\mu$-strongly convex iff $g(x) = f(x) - \frac{\mu}{2}x^2$ is convex, which for this $\mathcal{C}^2$ function requires

$$
g''(x) = 12x^2 - \mu \ge 0 \quad \text{for all } x
$$

At $x = 0$ this reads $-\mu \ge 0$, impossible for $\mu \gt 0$. Alternatively: strong convexity implies the quadratic lower bound $f(x) \ge f(0) + f'(0)x + \frac{\mu}{2}x^2 = \frac{\mu}{2}x^2$, but $x^4 \lt \frac{\mu}{2}x^2$ for $0 \lt \lvert x\rvert \lt \sqrt{\mu/2}$.

$$
\boxed{x^4 \text{ is strictly convex, yet } \mu\text{-strongly convex for no } \mu \gt 0}
$$

> **Key takeaway:** Strong convexity is a *uniform* curvature guarantee; a single flat point destroys it even when strict convexity survives.

In [9]:
# Problem L1.4 - x^4 is strictly convex yet dominated by every parabola near 0
xs = rng.uniform(-3, 3, 5000)
ys = rng.uniform(-3, 3, 5000)
th = rng.uniform(0.01, 0.99, 5000)
gap = th * xs**4 + (1 - th) * ys**4 - (th * xs + (1 - th) * ys) ** 4
distinct = np.abs(xs - ys) > 1e-6
print(f"minimum strict-convexity gap over distinct pairs : {gap[distinct].min():.3e}  (> 0)")
assert gap[distinct].min() > 0

print("\nstrong convexity fails for every mu > 0:")
for mu in (1.0, 0.1, 0.01):
    x_t = np.sqrt(mu / 2) / 2
    print(f"  mu = {mu:5.2f}: at x = {x_t:.4f},  x^4 = {x_t**4:.3e} < (mu/2)x^2 = {mu / 2 * x_t**2:.3e}")
    assert x_t**4 < mu / 2 * x_t**2

minimum strict-convexity gap over distinct pairs : 7.571e-08  (> 0)

strong convexity fails for every mu > 0:
  mu =  1.00: at x = 0.3536,  x^4 = 1.563e-02 < (mu/2)x^2 = 6.250e-02
  mu =  0.10: at x = 0.1118,  x^4 = 1.562e-04 < (mu/2)x^2 = 6.250e-04
  mu =  0.01: at x = 0.0354,  x^4 = 1.562e-06 < (mu/2)x^2 = 6.250e-06


### Problem L1.5 — Jensen's Inequality Implies AM–GM

**Problem Statement:** Using the concavity of $\ln$, prove the weighted arithmetic-geometric mean inequality: for $a_i \gt 0$ and weights $\theta_i \ge 0$ with $\sum_i \theta_i = 1$,

$$
\prod_{i=1}^{k} a_i^{\theta_i} \le \sum_{i=1}^{k} \theta_i a_i
$$

*Intuition:* The logarithm turns products into sums; concavity (Jensen upside down) then compares the two means.

**Solution:**

**Step 1 (Concavity of $\ln$).** On $(0, \infty)$, $\frac{d^2}{dx^2}\ln x = -x^{-2} \lt 0$, so $\ln$ is (strictly) concave and Jensen's inequality holds with the direction reversed:

$$
\ln\!\left(\sum_{i=1}^{k}\theta_i a_i\right) \ge \sum_{i=1}^{k}\theta_i \ln a_i
$$

**Step 2 (Exponentiate).** The right side equals $\ln\prod_i a_i^{\theta_i}$ by the logarithm laws. Since $\exp$ is increasing, applying it to both sides preserves the inequality:

$$
\sum_{i=1}^{k}\theta_i a_i \ge \prod_{i=1}^{k} a_i^{\theta_i}
$$

**Step 3 (Equality case).** Strict concavity of $\ln$ makes Jensen strict unless all the $a_i$ with $\theta_i \gt 0$ coincide; hence equality iff all weighted entries are equal. Taking $\theta_i = \frac{1}{k}$ recovers the classical AM–GM

$$
\boxed{\sqrt[k]{a_1 a_2 \cdots a_k} \le \frac{a_1 + a_2 + \cdots + a_k}{k}, \quad \text{equality iff } a_1 = \cdots = a_k}
$$

> **Key takeaway:** Many classical inequalities (AM–GM, Cauchy–Schwarz via $x^2$, Hölder via $x^p$) are Jensen's inequality applied to a well-chosen convex or concave function.

In [10]:
# Problem L1.5 - the weighted AM-GM inequality on random data
a = rng.uniform(0.1, 5.0, size=(2000, 4))
weights = rng.dirichlet(np.ones(4), size=2000)
gm = np.exp(np.sum(weights * np.log(a), axis=1))
am = np.sum(weights * a, axis=1)
print(f"violations of prod a_i^theta_i <= sum theta_i a_i : {int(np.sum(gm > am + 1e-12))} of 2000")
print(f"largest gap observed                              : {np.max(am - gm):.4f}")
equal = np.full(4, 3.0)
print(f"equality case a = (3,3,3,3): GM = {np.exp(np.mean(np.log(equal))):.6f}, AM = {equal.mean():.6f}")
assert np.all(gm <= am + 1e-12)

violations of prod a_i^theta_i <= sum theta_i a_i : 0 of 2000
largest gap observed                              : 1.5139
equality case a = (3,3,3,3): GM = 3.000000, AM = 3.000000


### Problem L1.6 — A Two-Variable Hessian Test

**Problem Statement:** Determine whether $f(x, y) = x^2 + xy + y^2$ is convex, strictly convex, and strongly convex on $\mathbb{R}^2$.

*Intuition:* A quadratic form is a bowl, a trough, or a saddle according to the eigenvalues of its (constant) Hessian.

**Solution:**

**Step 1 (Compute the Hessian).** $\nabla f = (2x + y, \ x + 2y)$ and

$$
\nabla^2 f = \begin{bmatrix} 2 & 1 \\ 1 & 2 \end{bmatrix} \quad \text{(constant in } (x,y)\text{)}
$$

**Step 2 (Eigenvalues).** The characteristic polynomial is $(2-\lambda)^2 - 1 = 0$, so $\lambda \in \{1, 3\}$. Both eigenvalues are strictly positive, hence $\nabla^2 f \succ 0$ everywhere. (Sylvester check: leading minors $2 \gt 0$ and $4 - 1 = 3 \gt 0$.)

**Step 3 (Classify).** A positive definite Hessian everywhere gives convexity and indeed strict convexity. For strong convexity, the best constant is the smallest eigenvalue: for all $\mathbf{v}$,

$$
\mathbf{v}^\top \nabla^2 f\, \mathbf{v} \ge \lambda_{\min}\lVert\mathbf{v}\rVert^2 = 1 \cdot \lVert\mathbf{v}\rVert^2
$$

so $\nabla^2 f \succeq 1 \cdot I$ and $f$ is $1$-strongly convex (and no better, witnessed by the eigenvector $(1, -1)$ of $\lambda = 1$).

$$
\boxed{f \text{ is convex, strictly convex, and } \mu\text{-strongly convex with sharp } \mu = \lambda_{\min} = 1}
$$

> **Key takeaway:** For quadratics the whole convexity hierarchy is read off the Hessian spectrum: $\lambda_{\min} \ge 0$ convex, $\lambda_{\min} \gt 0$ strictly and strongly convex with $\mu = \lambda_{\min}$.

In [11]:
# Problem L1.6 - eigenvalues, leading minors and the sharp modulus mu = 1
H = np.array([[2.0, 1.0], [1.0, 2.0]])
eig = np.linalg.eigvalsh(H)
print(f"eigenvalues        : {eig}          (hand: 1 and 3)")
print(f"leading minors     : {H[0, 0]:.4f}, {np.linalg.det(H):.4f}   (hand: 2 and 3)")
v = np.array([1.0, -1.0]) / np.sqrt(2.0)
print(f"v^T H v at v = (1,-1)/sqrt(2) : {v @ H @ v:.6f} = lambda_min, so mu = 1 is sharp")
assert np.allclose(eig, [1.0, 3.0]) and abs(v @ H @ v - 1.0) < 1e-12

eigenvalues        : [1. 3.]          (hand: 1 and 3)
leading minors     : 2.0000, 3.0000   (hand: 2 and 3)
v^T H v at v = (1,-1)/sqrt(2) : 1.000000 = lambda_min, so mu = 1 is sharp


## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — Convexity of Least Squares

**Problem Statement:** For $A \in \mathbb{R}^{m \times n}$ and $\mathbf{b} \in \mathbb{R}^m$, show that $f(\mathbf{w}) = \frac{1}{2}\lVert A\mathbf{w} - \mathbf{b}\rVert_2^2$ is convex, and determine exactly when it is strictly convex.

*Intuition:* Least squares is a paraboloid lifted through a linear map; flat directions appear exactly where the map has a nullspace.

**Solution:**

**Step 1 (Gradient and Hessian).** Expand $f(\mathbf{w}) = \frac{1}{2}\mathbf{w}^\top A^\top A\,\mathbf{w} - \mathbf{b}^\top A\,\mathbf{w} + \frac{1}{2}\lVert\mathbf{b}\rVert_2^2$. Differentiating,

$$
\nabla f(\mathbf{w}) = A^\top(A\mathbf{w} - \mathbf{b}), \qquad \nabla^2 f(\mathbf{w}) = A^\top A
$$

**Step 2 (PSD Hessian).** For any $\mathbf{v} \in \mathbb{R}^n$,

$$
\mathbf{v}^\top A^\top A\,\mathbf{v} = (A\mathbf{v})^\top(A\mathbf{v}) = \lVert A\mathbf{v}\rVert_2^2 \ge 0
$$

so $A^\top A \succeq 0$ and $f$ is convex by the second-order characterization — for *every* data matrix, with no assumptions.

**Step 3 (Strict convexity).** The quadratic $f$ is strictly convex iff $A^\top A \succ 0$, i.e. iff $\lVert A\mathbf{v}\rVert_2^2 \gt 0$ for all $\mathbf{v} \neq \mathbf{0}$, i.e. iff $\ker(A) = \{\mathbf{0}\}$ — full column rank, $\operatorname{rank}(A) = n$. If $\operatorname{rank}(A) \lt n$ (e.g. collinear features, or $m \lt n$), the minimizer set is the affine subspace of solutions of the normal equations $A^\top A\,\mathbf{w} = A^\top \mathbf{b}$, all attaining the same loss.

$$
\boxed{\nabla^2 f = A^\top A \succeq 0 \ \text{always}; \quad f \text{ strictly convex} \iff \operatorname{rank}(A) = n}
$$

> **Key takeaway:** Least squares is unconditionally convex, and identifiability of the model (full column rank) is exactly strict convexity of the training loss.

In [12]:
# Problem L2.1 - A^T A is always PSD; strict convexity needs full column rank
A_full = rng.normal(size=(8, 3))
A_def = A_full.copy()
A_def[:, 2] = A_def[:, 0] - 2.0 * A_def[:, 1]
for name, M in [("full column rank", A_full), ("rank deficient  ", A_def)]:
    eig = np.linalg.eigvalsh(M.T @ M)
    print(f"{name}: rank = {np.linalg.matrix_rank(M)}, eig(A^T A) = {eig}, "
          f"strictly convex = {eig.min() > 1e-10}")
assert np.linalg.eigvalsh(A_full.T @ A_full).min() > 1e-10
assert np.linalg.eigvalsh(A_def.T @ A_def).min() < 1e-10

full column rank: rank = 3, eig(A^T A) = [ 1.8502 13.3212 18.1431], strictly convex = True
rank deficient  : rank = 2, eig(A^T A) = [ 0.      4.8799 66.0433], strictly convex = False


### Problem L2.2 — Convexity of the Logistic Loss

**Problem Statement:** Show that the logistic loss for one sample, $\ell(\mathbf{w}) = \log\big(1 + e^{-y\,\mathbf{w}^\top \mathbf{x}}\big)$ with label $y \in \{-1, +1\}$, is a convex function of $\mathbf{w}$.

*Intuition:* Reduce to a scalar function of the margin $t = y\,\mathbf{w}^\top \mathbf{x}$ and check its curvature; affine composition then lifts convexity to $\mathbf{w}$.

**Solution:**

**Step 1 (Scalar reduction).** Write $\ell(\mathbf{w}) = \varphi(y\,\mathbf{w}^\top \mathbf{x})$ where $\varphi(t) = \log(1 + e^{-t})$. The map $\mathbf{w} \mapsto y\,\mathbf{w}^\top \mathbf{x}$ is linear (hence affine).

**Step 2 (Curvature of $\varphi$).** With the sigmoid $\sigma(t) = \frac{1}{1 + e^{-t}}$,

$$
\varphi'(t) = \frac{-e^{-t}}{1 + e^{-t}} = -(1 - \sigma(t)) = \sigma(t) - 1, \qquad \varphi''(t) = \sigma'(t) = \sigma(t)\big(1 - \sigma(t)\big)
$$

Since $0 \lt \sigma(t) \lt 1$ for all $t$, we get $\varphi''(t) \gt 0$: $\varphi$ is (strictly) convex on $\mathbb{R}$.

**Step 3 (Affine composition).** If $\varphi$ is convex and $T(\mathbf{w}) = \mathbf{c}^\top \mathbf{w}$ is affine, then for any $\mathbf{w}_1, \mathbf{w}_2$ and $\theta \in [0,1]$,

$$
\varphi\big(T(\theta\mathbf{w}_1 + (1-\theta)\mathbf{w}_2)\big) = \varphi\big(\theta\,T(\mathbf{w}_1) + (1-\theta)\,T(\mathbf{w}_2)\big) \le \theta\,\varphi(T(\mathbf{w}_1)) + (1-\theta)\,\varphi(T(\mathbf{w}_2))
$$

so $\ell = \varphi \circ T$ is convex. A finite sum over samples (nonnegative weights) preserves convexity, so the full training loss is convex; note $\ell$ is *not strictly* convex in $\mathbf{w}$ when $n \gt 1$, since it is constant along directions orthogonal to $\mathbf{x}$.

$$
\boxed{\ell(\mathbf{w}) = \log\big(1 + e^{-y\,\mathbf{w}^\top \mathbf{x}}\big) \text{ is convex; hence logistic training finds a global optimum}}
$$

> **Key takeaway:** Check curvature once in one dimension, then let affine composition and nonnegative sums carry convexity to the full parameter space — the standard pattern for ML losses.

In [13]:
# Problem L2.2 - phi'' > 0 for the logistic loss, and the chord test in w
sigma = lambda t: 1.0 / (1.0 + np.exp(-t))
ts = np.linspace(-12, 12, 2401)
print(f"min phi''(t) = min sigma(t)(1 - sigma(t)) on [-12, 12] : {np.min(sigma(ts) * (1 - sigma(ts))):.3e}  (> 0)")

x_feat, y_lab = rng.normal(size=4), 1.0
ell = lambda w: np.log1p(np.exp(-y_lab * (w @ x_feat)))
w1, w2 = rng.normal(size=(3000, 4)), rng.normal(size=(3000, 4))
th = rng.uniform(0, 1, 3000)
gap = np.array([t * ell(a) + (1 - t) * ell(b) - ell(t * a + (1 - t) * b)
                for a, b, t in zip(w1, w2, th)])
print(f"min chord gap of w -> log(1 + exp(-y w^T x)) over 3000 triples : {gap.min():.3e}  (>= 0)")
assert np.min(sigma(ts) * (1 - sigma(ts))) > 0 and gap.min() >= -1e-12

min phi''(t) = min sigma(t)(1 - sigma(t)) on [-12, 12] : 6.144e-06  (> 0)
min chord gap of w -> log(1 + exp(-y w^T x)) over 3000 triples : 1.189e-08  (>= 0)


### Problem L2.3 — Ridge Regularization and Strong Convexity

**Problem Statement:** Let $L(\mathbf{w})$ be any convex, twice-differentiable loss. Show that the ridge-regularized objective $f(\mathbf{w}) = L(\mathbf{w}) + \frac{\lambda}{2}\lVert\mathbf{w}\rVert_2^2$ with $\lambda \gt 0$ is $\lambda$-strongly convex, and conclude that it has a unique minimizer whenever one exists. For least squares, quantify the effect on the Hessian spectrum.

*Intuition:* Adding a bowl of curvature $\lambda$ to a possibly flat landscape guarantees curvature at least $\lambda$ everywhere.

**Solution:**

**Step 1 (Strong convexity by definition).** $f$ is $\lambda$-strongly convex iff $f(\mathbf{w}) - \frac{\lambda}{2}\lVert\mathbf{w}\rVert_2^2 = L(\mathbf{w})$ is convex — which it is by hypothesis. Done. Equivalently, at the Hessian level,

$$
\nabla^2 f(\mathbf{w}) = \nabla^2 L(\mathbf{w}) + \lambda I \succeq 0 + \lambda I = \lambda I
$$

since adding $\lambda I$ shifts every eigenvalue up by $\lambda$.

**Step 2 (Uniqueness).** Strong convexity implies strict convexity (the added quadratic makes every chord inequality strict), and a strictly convex function has at most one minimizer: if $\mathbf{w}_1 \neq \mathbf{w}_2$ both attained the minimum $p^{\star}$, the midpoint would satisfy $f(\mathbf{m}) \lt \frac{1}{2}p^{\star} + \frac{1}{2}p^{\star} = p^{\star}$, a contradiction. (In fact strong convexity also gives coercivity, hence existence — so the minimizer exists and is unique unconditionally.)

**Step 3 (Least-squares spectrum).** For $L(\mathbf{w}) = \frac{1}{2}\lVert A\mathbf{w} - \mathbf{b}\rVert_2^2$, the regularized Hessian is $A^\top A + \lambda I$ with eigenvalues $\sigma_i^2 + \lambda \gt 0$ (where $\sigma_i$ are the singular values of $A$, some possibly zero). The problem becomes well-posed even for rank-deficient $A$, with unique solution $\mathbf{w}^{\star} = (A^\top A + \lambda I)^{-1}A^\top \mathbf{b}$, and the condition number improves:

$$
\kappa_\lambda = \frac{\sigma_{\max}^2 + \lambda}{\sigma_{\min}^2 + \lambda} \quad \text{is strictly decreasing in } \lambda \gt 0, \quad \text{with } \kappa_0 = \frac{\sigma_{\max}^2}{\sigma_{\min}^2} \ (= +\infty \text{ if } \sigma_{\min} = 0)
$$

$$
\boxed{f = L + \tfrac{\lambda}{2}\lVert\cdot\rVert_2^2 \ \text{ is } \lambda\text{-strongly convex with a unique minimizer for every convex } L}
$$

> **Key takeaway:** Regularization is not merely a statistical prior — it is a convexity upgrade that buys existence, uniqueness, and better conditioning simultaneously.

In [14]:
# Problem L2.3 - the ridge Hessian spectrum and the condition number kappa_lambda
A = rng.normal(size=(6, 3))
A[:, 2] = A[:, 0] + A[:, 1]                      # rank deficient on purpose
print("singular values of A :", np.linalg.svd(A, compute_uv=False))

kappas = []
for lam in (0.0, 0.01, 0.1, 1.0, 10.0):
    eig = np.linalg.eigvalsh(A.T @ A + lam * np.eye(3))
    kappa = np.inf if eig.min() <= 1e-12 else eig.max() / eig.min()
    kappas.append(kappa)
    print(f"  lambda = {lam:6.2f}: eig_min = {eig.min():.4e}, kappa = {kappa:.4e}")
print("kappa_lambda strictly decreasing for lambda > 0 :",
      all(kappas[i] > kappas[i + 1] for i in range(1, len(kappas) - 1)))
assert kappas[0] == np.inf
assert all(kappas[i] > kappas[i + 1] for i in range(1, len(kappas) - 1))

singular values of A : [4.614  1.9053 0.    ]
  lambda =   0.00: eig_min = 1.4937e-15, kappa = inf
  lambda =   0.01: eig_min = 1.0000e-02, kappa = 2.1299e+03
  lambda =   0.10: eig_min = 1.0000e-01, kappa = 2.1389e+02
  lambda =   1.00: eig_min = 1.0000e+00, kappa = 2.2289e+01
  lambda =  10.00: eig_min = 1.0000e+01, kappa = 3.1289e+00
kappa_lambda strictly decreasing for lambda > 0 : True


### Problem L2.4 — Entropy, Gibbs' Inequality, and the Boltzmann Distribution

**Problem Statement:** (a) Show that the negative entropy $f(\mathbf{p}) = \sum_{i=1}^n p_i \log p_i$ is convex on the probability simplex. (b) Using Jensen's inequality, prove Gibbs' inequality: $D_{\mathrm{KL}}(\mathbf{p} \parallel \mathbf{q}) = \sum_i p_i \log\frac{p_i}{q_i} \ge 0$ with equality iff $\mathbf{p} = \mathbf{q}$. (c) **Statistical mechanics.** A system has energy levels $E_1, \dots, E_n$ and is known to have mean energy $\bar{E}$. Show that among all distributions satisfying $\sum_i p_i E_i = \bar{E}$ the entropy is maximized by the Boltzmann distribution $p_i^{\star} = e^{-\beta E_i}/Z(\beta)$ with $Z(\beta) = \sum_j e^{-\beta E_j}$ and $\beta$ chosen to meet the constraint, and that this maximizer is unique.

*Intuition:* Entropy's concavity is one-dimensional curvature summed coordinate-wise; KL nonnegativity is Jensen applied to the concave logarithm of a likelihood ratio; and once (b) is available, the Boltzmann distribution wins the entropy contest without a single derivative being taken.

**Solution:**

**Step 1 (Negative entropy is convex).** Each summand $\varphi(p) = p\log p$ has $\varphi''(p) = \frac{1}{p} \gt 0$ on $(0, 1]$, so each is convex; a sum of convex functions of separate coordinates is convex (its Hessian is $\operatorname{diag}(1/p_1, \dots, 1/p_n) \succ 0$ on the interior). Restricting to the simplex (a convex set) preserves convexity. Hence entropy $H(\mathbf{p}) = -f(\mathbf{p})$ is concave, and maximum-entropy problems are convex programs.

**Step 2 (Gibbs via Jensen).** Assume $p_i, q_i \gt 0$. Write minus the divergence as an expectation of a log-ratio under $\mathbf{p}$:

$$
-D_{\mathrm{KL}}(\mathbf{p} \parallel \mathbf{q}) = \sum_i p_i \log\frac{q_i}{p_i} \le \log\!\left(\sum_i p_i\,\frac{q_i}{p_i}\right) = \log\!\left(\sum_i q_i\right) = \log 1 = 0
$$

where the inequality is Jensen for the concave $\log$ with weights $p_i$ and points $\frac{q_i}{p_i}$.

**Step 3 (Equality case).** $\log$ is *strictly* concave, so Jensen is an equality iff all points coincide: $\frac{q_i}{p_i} = c$ for all $i$; summing gives $c = 1$, i.e. $\mathbf{p} = \mathbf{q}$.

**Step 4 (c: the Boltzmann distribution maximizes entropy).** The feasible set $\{\mathbf{p} \mid p_i \ge 0, \ \sum_i p_i = 1, \ \sum_i p_i E_i = \bar{E}\}$ is an intersection of a hyperplane, a hyperplane, and the nonnegative orthant, hence a polyhedron and convex; the objective $-H$ is convex by Step 1, so this is a convex problem. Let $\mathbf{q}$ be any feasible distribution. Because $\log p_i^{\star} = -\beta E_i - \log Z(\beta)$,

$$
-\sum_i q_i \log p_i^{\star} = \beta \sum_i q_i E_i + \log Z(\beta) = \beta\bar{E} + \log Z(\beta)
$$

which is the *same number* for every feasible $\mathbf{q}$, since only the constraint value $\bar{E}$ enters. Now apply part (b):

$$
0 \le D_{\mathrm{KL}}(\mathbf{q} \parallel \mathbf{p}^{\star}) = \sum_i q_i \log q_i - \sum_i q_i \log p_i^{\star} = -H(\mathbf{q}) + \beta\bar{E} + \log Z(\beta)
$$

so $H(\mathbf{q}) \le \log Z(\beta) + \beta\bar{E}$ for every feasible $\mathbf{q}$, with equality precisely when $\mathbf{q} = \mathbf{p}^{\star}$. Since $\mathbf{p}^{\star}$ is itself feasible by the choice of $\beta$ and attains that value, it is the unique maximizer. No Lagrange multipliers were needed: the whole argument is Gibbs' inequality plus the fact that the constraint fixes $\sum_i q_i E_i$.

$$
\boxed{D_{\mathrm{KL}}(\mathbf{p} \parallel \mathbf{q}) \ge 0 \text{ with equality iff } \mathbf{p} = \mathbf{q}; \quad \operatorname{argmax}\{H(\mathbf{q}) : \textstyle\sum_i q_i E_i = \bar{E}\} = \big(e^{-\beta E_i}/Z(\beta)\big)_i}
$$

> **Key takeaway:** Gibbs' inequality is a two-line consequence of Jensen's, and it immediately delivers the Boltzmann distribution of statistical mechanics as the unique entropy maximizer at fixed mean energy.

In [15]:
# Problem L2.4 - Gibbs' inequality and concavity of the entropy
P = rng.dirichlet(np.ones(5), size=4000)
Qd = rng.dirichlet(np.ones(5), size=4000)
kl = np.sum(P * np.log(P / Qd), axis=1)
print(f"min KL over 4000 random pairs : {kl.min():.3e}  (>= 0)")
p0 = np.array([0.2, 0.5, 0.3])
print(f"KL(p || p)                    : {np.sum(p0 * np.log(p0 / p0)):.3e}  (equality case)")

H = lambda p: -np.sum(p * np.log(p))
pa, pb, t = rng.dirichlet(np.ones(3)), rng.dirichlet(np.ones(3)), 0.4
print(f"entropy concavity: H(mix) = {H(t * pa + (1 - t) * pb):.6f} >= "
      f"{t * H(pa) + (1 - t) * H(pb):.6f} = mix of entropies")
assert kl.min() >= -1e-15 and H(t * pa + (1 - t) * pb) >= t * H(pa) + (1 - t) * H(pb)

# part (c): a three-level system with energies 0, 1, 2 at inverse temperature beta = 0.5
E_lvl, beta = np.array([0.0, 1.0, 2.0]), 0.5
Z = np.exp(-beta * E_lvl).sum()
p_boltz = np.exp(-beta * E_lvl) / Z
E_bar = p_boltz @ E_lvl
print(f"\nBoltzmann p* = {p_boltz},  mean energy = {E_bar:.6f}")
print(f"H(p*) = {H(p_boltz):.6f}   log Z + beta*E_bar = {np.log(Z) + beta * E_bar:.6f}")
assert abs(H(p_boltz) - (np.log(Z) + beta * E_bar)) < 1e-12

# feasible perturbations keep sum = 1 and mean energy = E_bar: direction (1, -2, 1)
direction = np.array([1.0, -2.0, 1.0])
shifts = np.linspace(-0.08, 0.08, 17)
family = p_boltz + np.outer(shifts, direction)
entropies = np.array([H(q) for q in family])
print(f"all perturbations feasible : {np.allclose(family.sum(1), 1.0) and np.allclose(family @ E_lvl, E_bar) and family.min() > 0}")
print(f"entropy is maximal at shift : {shifts[entropies.argmax()]:.3f}   (predicted 0)")
print(f"largest competitor falls short by : {H(p_boltz) - entropies[shifts != 0].max():.3e}")
assert np.allclose(family.sum(1), 1.0) and np.allclose(family @ E_lvl, E_bar)
assert entropies.argmax() == np.argmin(np.abs(shifts)) and H(p_boltz) > entropies[shifts != 0].max()

min KL over 4000 random pairs : 8.493e-03  (>= 0)
KL(p || p)                    : 0.000e+00  (equality case)
entropy concavity: H(mix) = 1.096441 >= 0.884985 = mix of entropies

Boltzmann p* = [0.5065 0.3072 0.1863],  mean energy = 0.679843
H(p*) = 1.020191   log Z + beta*E_bar = 1.020191
all perturbations feasible : True
entropy is maximal at shift : 0.000   (predicted 0)
largest competitor falls short by : 1.010e-03


### Problem L2.5 — Equilibrium of a Spring Chain Is a Convex Problem

**Problem Statement:** Two springs with stiffnesses $k_1, k_2 \gt 0$ connect a wall to mass 1 and mass 1 to mass 2; an external force $F$ pulls mass 2. With displacements $x_1, x_2$, the potential energy is

$$
U(x_1, x_2) = \frac{1}{2}k_1 x_1^2 + \frac{1}{2}k_2 (x_2 - x_1)^2 - F x_2
$$

Show $U$ is strictly convex, find the equilibrium, and explain why it is the unique stable configuration.

*Intuition:* Hooke-law energies are quadratic bowls; coupling them keeps the total energy a bowl as long as every stiffness is positive.

**Solution:**

**Step 1 (Hessian).** $\nabla U = \big(k_1 x_1 - k_2(x_2 - x_1), \ k_2(x_2 - x_1) - F\big)$ and the (constant) stiffness matrix is

$$
K = \nabla^2 U = \begin{bmatrix} k_1 + k_2 & -k_2 \\ -k_2 & k_2 \end{bmatrix}
$$

**Step 2 (Positive definiteness).** Sylvester's criterion: the leading minors are $k_1 + k_2 \gt 0$ and $\det K = (k_1 + k_2)k_2 - k_2^2 = k_1 k_2 \gt 0$. Hence $K \succ 0$ and $U$ is strictly (indeed strongly) convex, so any stationary point is the unique global energy minimum.

**Step 3 (Equilibrium).** Solve $\nabla U = \mathbf{0}$: the second equation gives $k_2(x_2 - x_1) = F$, and substituting into the first gives $k_1 x_1 = F$. Hence

$$
x_1^{\star} = \frac{F}{k_1}, \qquad x_2^{\star} = \frac{F}{k_1} + \frac{F}{k_2}
$$

— the compliances add in series, recovering the classical formula $\frac{1}{k_{\text{eff}}} = \frac{1}{k_1} + \frac{1}{k_2}$.

**Step 4 (Stability).** By the Lagrange–Dirichlet principle, a strict minimum of potential energy is a stable equilibrium: any perturbation raises $U$, and the restoring force $-\nabla U$ points back toward $\mathbf{x}^{\star}$. Convexity guarantees there are no other equilibria to fall into.

$$
\boxed{K \succ 0 \implies \text{unique stable equilibrium } x_1^{\star} = \tfrac{F}{k_1}, \ x_2^{\star} = \tfrac{F}{k_1} + \tfrac{F}{k_2}}
$$

> **Key takeaway:** Positive stiffness makes elasticity a strongly convex optimization problem — which is precisely why structures settle into one predictable shape.

In [16]:
# Problem L2.5 - the spring chain with k1 = 2, k2 = 3, F = 6
k1, k2, F = 2.0, 3.0, 6.0
K = np.array([[k1 + k2, -k2], [-k2, k2]])
U = lambda x: 0.5 * k1 * x[0]**2 + 0.5 * k2 * (x[1] - x[0])**2 - F * x[1]
grad_U = lambda x: np.array([k1 * x[0] - k2 * (x[1] - x[0]), k2 * (x[1] - x[0]) - F])

x_star = np.array([F / k1, F / k1 + F / k2])
print(f"stiffness matrix K   : {K.tolist()}")
print(f"eigenvalues of K     : {np.linalg.eigvalsh(K)}   det K = k1 k2 = {np.linalg.det(K):.4f}")
print(f"predicted equilibrium: {x_star}   (hand: F/k1 = 3, F/k1 + F/k2 = 5)")
print(f"gradient there       : {grad_U(x_star)}")
print(f"solving K x = (0, F) : {np.linalg.solve(K, np.array([0.0, F]))}")

perturbed = x_star + rng.normal(scale=0.5, size=(2000, 2))
below = int(np.sum([U(p) < U(x_star) for p in perturbed]))
print(f"perturbations with energy below U(x*) : {below} of 2000  (strict minimum)")
assert np.allclose(grad_U(x_star), 0.0) and np.linalg.eigvalsh(K).min() > 0 and below == 0

stiffness matrix K   : [[5.0, -3.0], [-3.0, 3.0]]
eigenvalues of K     : [0.8377 7.1623]   det K = k1 k2 = 6.0000
predicted equilibrium: [3. 5.]   (hand: F/k1 = 3, F/k1 + F/k2 = 5)
gradient there       : [0. 0.]
solving K x = (0, F) : [3. 5.]
perturbations with energy below U(x*) : 0 of 2000  (strict minimum)


### Problem L2.6 — Markowitz Portfolio Variance Is Convex

**Problem Statement:** Let $\mathbf{r}$ be a random return vector with mean $\boldsymbol{\mu}$ and covariance $\Sigma = \mathbb{E}\big[(\mathbf{r}-\boldsymbol{\mu})(\mathbf{r}-\boldsymbol{\mu})^\top \big]$. Show that the portfolio variance $f(\mathbf{w}) = \mathbf{w}^\top \Sigma\,\mathbf{w}$ is convex in the weights $\mathbf{w}$, and that the Markowitz feasible set $\{\mathbf{w} \mid \mathbf{1}^\top \mathbf{w} = 1, \ \boldsymbol{\mu}^\top \mathbf{w} \ge r_0\}$ is convex. Classify the resulting problem.

*Intuition:* A covariance matrix is an expectation of rank-one squares, hence PSD; linear budget and return constraints carve out a polyhedron.

**Solution:**

**Step 1 (Covariance is PSD).** For any $\mathbf{w}$,

$$
\mathbf{w}^\top \Sigma\,\mathbf{w} = \mathbb{E}\big[\mathbf{w}^\top(\mathbf{r}-\boldsymbol{\mu})(\mathbf{r}-\boldsymbol{\mu})^\top \mathbf{w}\big] = \mathbb{E}\big[\big((\mathbf{r}-\boldsymbol{\mu})^\top \mathbf{w}\big)^2\big] \ge 0
$$

so $\Sigma \succeq 0$. Note $\mathbf{w}^\top \Sigma\mathbf{w} = \operatorname{Var}(\mathbf{r}^\top \mathbf{w})$: the objective *is* the variance of the portfolio return.

**Step 2 (Convexity of the objective).** $f(\mathbf{w}) = \mathbf{w}^\top \Sigma\mathbf{w}$ has constant Hessian $\nabla^2 f = 2\Sigma \succeq 0$, so $f$ is convex by the second-order characterization; it is strictly convex iff $\Sigma \succ 0$ (no riskless nontrivial combination of assets).

**Step 3 (Convexity of the feasible set).** $\{\mathbf{1}^\top \mathbf{w} = 1\}$ is a hyperplane and $\{\boldsymbol{\mu}^\top \mathbf{w} \ge r_0\}$ is a half-space; their intersection is convex (a polyhedron). Adding no-short-sale bounds $\mathbf{w} \ge \mathbf{0}$ keeps it polyhedral.

**Step 4 (Classification).** Convex quadratic objective, linear constraints: a **quadratic program (QP)** — continuous, convex, smooth, constrained in the taxonomy. Hence every local optimum is global, and the mean-variance frontier can be traced reliably by varying $r_0$.

$$
\boxed{\min_{\mathbf{w}} \mathbf{w}^\top \Sigma\,\mathbf{w} \ \text{ s.t. } \mathbf{1}^\top \mathbf{w} = 1, \ \boldsymbol{\mu}^\top \mathbf{w} \ge r_0 \ \text{ is a convex QP}}
$$

> **Key takeaway:** Risk minimization is convex because covariance matrices are PSD by construction — a structural fact about expectations of squares, not an assumption about markets.

In [17]:
# Problem L2.6 - portfolio variance: Sigma is PSD and w -> w^T Sigma w is convex
L = np.array([[1.0, 0.3, 0.0, 0.2], [0.0, 0.9, 0.1, 0.0],
              [0.0, 0.0, 0.7, 0.4], [0.0, 0.0, 0.0, 1.1]])
returns = rng.normal(size=(500, 4)) @ L
Sigma = np.cov(returns, rowvar=False)
print("eigenvalues of Sigma :", np.linalg.eigvalsh(Sigma))

w = rng.normal(size=4)
w = w / w.sum()                                   # budget constraint 1^T w = 1
print(f"w^T Sigma w        = {w @ Sigma @ w:.6f}")
print(f"sample Var(returns w) = {np.var(returns @ w, ddof=1):.6f}   (the same quantity)")

w1, w2 = rng.normal(size=(3000, 4)), rng.normal(size=(3000, 4))
th = rng.uniform(0, 1, 3000)
gap = np.array([t * (a @ Sigma @ a) + (1 - t) * (b @ Sigma @ b)
                - ((t * a + (1 - t) * b) @ Sigma @ (t * a + (1 - t) * b))
                for a, b, t in zip(w1, w2, th)])
print(f"min chord gap of the variance objective : {gap.min():.3e}  (>= 0)")
assert np.linalg.eigvalsh(Sigma).min() > -1e-12 and gap.min() >= -1e-12
assert abs(w @ Sigma @ w - np.var(returns @ w, ddof=1)) < 1e-9

eigenvalues of Sigma : [0.4173 0.7089 1.1199 1.7571]
w^T Sigma w        = 5.060088
sample Var(returns w) = 5.060088   (the same quantity)
min chord gap of the variance objective : 1.216e-03  (>= 0)


## L3 — Challenge Proofs

### Problem L3.1 — Convexity of Log-Sum-Exp

**Problem Statement:** Prove that $f(\mathbf{x}) = \log\big(\sum_{i=1}^n e^{x_i}\big)$ is convex on $\mathbb{R}^n$ by computing its Hessian and establishing $\mathbf{v}^\top \nabla^2 f(\mathbf{x})\,\mathbf{v} \ge 0$ via the Cauchy–Schwarz inequality. Explain its role as a smooth maximum.

*Intuition:* The gradient of log-sum-exp is the softmax distribution; its Hessian is that distribution's covariance, and covariances are PSD.

**Solution:**

**Step 1 (Gradient = softmax).** Let $Z(\mathbf{x}) = \sum_i e^{x_i}$ and $p_i = \frac{e^{x_i}}{Z}$, so $p_i \gt 0$ and $\sum_i p_i = 1$. Then

$$
\frac{\partial f}{\partial x_i} = \frac{e^{x_i}}{Z} = p_i
$$

**Step 2 (Hessian).** Differentiating again,

$$
\frac{\partial^2 f}{\partial x_i \partial x_j} = \frac{\delta_{ij} e^{x_i}}{Z} - \frac{e^{x_i} e^{x_j}}{Z^2} = \delta_{ij}\,p_i - p_i p_j, \qquad \nabla^2 f = \operatorname{diag}(\mathbf{p}) - \mathbf{p}\mathbf{p}^\top
$$

which is exactly the covariance matrix of a categorical random variable with distribution $\mathbf{p}$.

**Step 3 (PSD via Cauchy–Schwarz).** For any $\mathbf{v} \in \mathbb{R}^n$,

$$
\mathbf{v}^\top \nabla^2 f\,\mathbf{v} = \sum_i p_i v_i^2 - \Big(\sum_i p_i v_i\Big)^2
$$

Apply Cauchy–Schwarz to the vectors with components $a_i = \sqrt{p_i}$ and $b_i = \sqrt{p_i}\,v_i$:

$$
\Big(\sum_i p_i v_i\Big)^2 = \Big(\sum_i a_i b_i\Big)^2 \le \Big(\sum_i a_i^2\Big)\Big(\sum_i b_i^2\Big) = 1 \cdot \sum_i p_i v_i^2
$$

Hence $\mathbf{v}^\top \nabla^2 f\,\mathbf{v} \ge 0$ for all $\mathbf{v}$: the Hessian is PSD everywhere and $f$ is convex. (Equality holds iff $\mathbf{v}$ is constant, so $f$ is not strictly convex: it is affine along $\mathbf{1}$, indeed $f(\mathbf{x} + t\mathbf{1}) = f(\mathbf{x}) + t$.)

**Step 4 (Smooth max).** The bounds $\max_i x_i \le f(\mathbf{x}) \le \max_i x_i + \log n$ (lower: keep one term; upper: bound each term by the max) make log-sum-exp a smooth, convex surrogate for the max — the analytic engine of softmax classification and the free energy of statistical mechanics.

$$
\boxed{\nabla^2 \log\textstyle\sum_i e^{x_i} = \operatorname{diag}(\mathbf{p}) - \mathbf{p}\mathbf{p}^\top \succeq 0 \implies \text{log-sum-exp is convex}}
$$

> **Key takeaway:** Log-sum-exp is convex because its Hessian is a softmax covariance — a beautiful identity linking convex analysis, probability, and deep learning in one matrix.

In [18]:
# Problem L3.1 - the log-sum-exp Hessian is diag(p) - p p^T, and it is PSD
def lse(v):
    m = v.max()
    return m + np.log(np.exp(v - m).sum())

def softmax(v):
    m = v.max()
    e = np.exp(v - m)
    return e / e.sum()

x_pt = np.array([0.5, -1.0, 2.0])
p = softmax(x_pt)
H = np.diag(p) - np.outer(p, p)

eps, n = 1e-4, 3
H_num = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        ei, ej = np.zeros(n), np.zeros(n)
        ei[i], ej[j] = eps, eps
        H_num[i, j] = (lse(x_pt + ei + ej) - lse(x_pt + ei - ej)
                       - lse(x_pt - ei + ej) + lse(x_pt - ei - ej)) / (4 * eps * eps)
print("analytic Hessian diag(p) - p p^T :")
print(H)
print(f"max |analytic - finite difference| : {np.abs(H - H_num).max():.3e}")
print(f"eigenvalues                        : {np.linalg.eigvalsh(H)}  (PSD, one zero)")
print(f"H applied to the all-ones vector   : {H @ np.ones(3)}  (the flat direction)")
print(f"bounds: max x = {x_pt.max():.4f} <= lse = {lse(x_pt):.4f} <= max x + log 3 = {x_pt.max() + np.log(3):.4f}")
assert np.abs(H - H_num).max() < 1e-6 and np.linalg.eigvalsh(H).min() > -1e-12
assert x_pt.max() <= lse(x_pt) <= x_pt.max() + np.log(3) + 1e-12

analytic Hessian diag(p) - p p^T :
[[ 0.1446 -0.0069 -0.1377]
 [-0.0069  0.0376 -0.0307]
 [-0.1377 -0.0307  0.1684]]
max |analytic - finite difference| : 1.296e-08
eigenvalues                        : [0.     0.0546 0.296 ]  (PSD, one zero)
H applied to the all-ones vector   : [0. 0. 0.]  (the flat direction)
bounds: max x = 2.0000 <= lse = 2.2413 <= max x + log 3 = 3.0986


### Problem L3.2 — Quasiconvex but Not Convex

**Problem Statement:** A function $f$ is **quasiconvex** if every sublevel set $S_\alpha = \{x \mid f(x) \le \alpha\}$ is convex. (a) Show that every convex function is quasiconvex. (b) Show that $f(x) = \sqrt{\lvert x\rvert}$ on $\mathbb{R}$ is quasiconvex but not convex. (c) Show that quasiconvex functions can have non-convex-type behavior: exhibit a quasiconvex $f$ and points where Jensen's inequality fails.

*Intuition:* Quasiconvexity only constrains the shape of level regions (unimodality), not the curvature of the graph.

**Solution:**

**Step 1 (a: convex implies quasiconvex).** Let $f$ be convex, $x, y \in S_\alpha$, $\theta \in [0,1]$. Then

$$
f(\theta x + (1-\theta)y) \le \theta f(x) + (1-\theta) f(y) \le \theta\alpha + (1-\theta)\alpha = \alpha
$$

so $\theta x + (1-\theta)y \in S_\alpha$: every sublevel set is convex.

**Step 2 (b: sublevel sets of $\sqrt{\lvert x\rvert}$).** For $\alpha \lt 0$, $S_\alpha = \emptyset$ (convex). For $\alpha \ge 0$,

$$
S_\alpha = \{x \mid \sqrt{\lvert x\rvert} \le \alpha\} = [-\alpha^2, \ \alpha^2]
$$

an interval, hence convex. So $f$ is quasiconvex.

**Step 3 (b: convexity fails).** Test the chord between $x = 0$ and $y = 1$ at $\theta = \frac{1}{2}$:

$$
f\left(\frac{0+1}{2}\right) = \sqrt{\tfrac{1}{2}} \approx 0.707 \qquad \text{vs} \qquad \frac{f(0) + f(1)}{2} = \frac{0 + 1}{2} = 0.5
$$

Since $0.707 \gt 0.5$, the chord inequality is violated and $f$ is not convex (the graph is concave on each side of the origin).

**Step 4 (c: Jensen fails).** The same three points violate Jensen with weights $(\frac{1}{2}, \frac{1}{2})$, showing Jensen characterizes convexity, not quasiconvexity. Quasiconvex functions retain *unimodality* (useful: bisection on sublevel sets still finds global minima), but lose the calculus of convexity: sums of quasiconvex functions need not be quasiconvex, e.g. $\sqrt{\lvert x\rvert} + \sqrt{\lvert x - 1\rvert}$ has two separate strict local minima at $x = 0$ and $x = 1$.

$$
\boxed{\sqrt{\lvert x\rvert} \ \text{ is quasiconvex (interval sublevel sets) but not convex (chord test fails)}}
$$

> **Key takeaway:** Convexity is strictly stronger than quasiconvexity: sublevel-set geometry survives, but Jensen, sums, and duality theory all require the full chord inequality.

In [19]:
# Problem L3.2 - sqrt|x| has interval sublevel sets but fails the chord test
f_q = lambda x: np.sqrt(np.abs(x))
print(f"f(0.5) = {f_q(0.5):.6f}  vs  (f(0) + f(1))/2 = {(f_q(0.0) + f_q(1.0)) / 2:.6f}"
      f"  ->  chord lies below the graph")
assert f_q(0.5) > (f_q(0.0) + f_q(1.0)) / 2

grid = np.linspace(-4, 4, 40_001)
for alpha in (0.5, 1.0, 2.0):
    S = grid[f_q(grid) <= alpha]
    print(f"  sublevel set at alpha = {alpha}: [{S.min():.4f}, {S.max():.4f}]"
          f"   (predicted [{-alpha**2:.2f}, {alpha**2:.2f}], an interval)")
    assert abs(S.min() + alpha**2) < 1e-3 and abs(S.max() - alpha**2) < 1e-3

two = lambda x: np.sqrt(np.abs(x)) + np.sqrt(np.abs(x - 1.0))
print(f"\nsum of two quasiconvex pieces at 0, 0.5, 1 : "
      f"{two(0.0):.4f}, {two(0.5):.4f}, {two(1.0):.4f}  -> two separate minima")
assert two(0.0) < two(0.5) and two(1.0) < two(0.5)

f(0.5) = 0.707107  vs  (f(0) + f(1))/2 = 0.500000  ->  chord lies below the graph
  sublevel set at alpha = 0.5: [-0.2500, 0.2500]   (predicted [-0.25, 0.25], an interval)
  sublevel set at alpha = 1.0: [-1.0000, 1.0000]   (predicted [-1.00, 1.00], an interval)
  sublevel set at alpha = 2.0: [-4.0000, 4.0000]   (predicted [-4.00, 4.00], an interval)

sum of two quasiconvex pieces at 0, 0.5, 1 : 1.0000, 1.4142, 1.0000  -> two separate minima


### Problem L3.3 — The PSD Cone and Convexity of the Largest Eigenvalue

**Problem Statement:** On the space $\mathbb{S}^n$ of symmetric matrices: (a) prove that the PSD cone $\mathbb{S}^n_+ = \{X \mid X \succeq 0\}$ is a convex cone; (b) prove that $\lambda_{\max}(X)$ is a convex function of $X$, using the variational representation $\lambda_{\max}(X) = \sup_{\lVert\mathbf{v}\rVert_2 = 1} \mathbf{v}^\top X\mathbf{v}$.

*Intuition:* Each condition $\mathbf{v}^\top X\mathbf{v} \ge 0$ is a linear inequality in the entries of $X$; the PSD cone is an infinite intersection of such half-spaces, and the top eigenvalue is a supremum of linear functionals.

**Solution:**

**Step 1 (a: convex cone).** Let $X, Y \succeq 0$ and $\alpha, \beta \ge 0$. For every $\mathbf{v} \in \mathbb{R}^n$,

$$
\mathbf{v}^\top(\alpha X + \beta Y)\mathbf{v} = \alpha\,\mathbf{v}^\top X\mathbf{v} + \beta\,\mathbf{v}^\top Y\mathbf{v} \ge 0
$$

so $\alpha X + \beta Y \succeq 0$. Closure under nonnegative combinations gives both convexity (take $\alpha = \theta$, $\beta = 1-\theta$) and the cone property (take $\beta = 0$). Structurally, $\mathbb{S}^n_+ = \bigcap_{\mathbf{v}} \{X \mid \mathbf{v}^\top X\mathbf{v} \ge 0\}$ is an intersection of half-spaces in $\mathbb{S}^n$ — convex by the intersection theorem, and semidefinite programming optimizes over exactly this set.

**Step 2 (b: variational representation).** By the spectral theorem, $X = Q\Lambda Q^\top$ with orthonormal $Q$; for unit $\mathbf{v}$, writing $\mathbf{u} = Q^\top \mathbf{v}$ (also unit),

$$
\mathbf{v}^\top X\mathbf{v} = \sum_i \lambda_i u_i^2 \le \lambda_{\max} \sum_i u_i^2 = \lambda_{\max}(X)
$$

with equality at the top eigenvector. Hence $\lambda_{\max}(X) = \sup_{\lVert\mathbf{v}\rVert_2=1} \mathbf{v}^\top X\mathbf{v}$.

**Step 3 (b: supremum of linear maps).** For each fixed unit $\mathbf{v}$, the map $X \mapsto \mathbf{v}^\top X\mathbf{v} = \operatorname{tr}(X\,\mathbf{v}\mathbf{v}^\top)$ is *linear* in $X$. A pointwise supremum of linear (hence convex) functions is convex: for $X, Y \in \mathbb{S}^n$ and $\theta \in [0,1]$,

$$
\mathbf{v}^\top \big(\theta X + (1-\theta)Y\big)\mathbf{v} = \theta\,\mathbf{v}^\top X\mathbf{v} + (1-\theta)\,\mathbf{v}^\top Y\mathbf{v} \le \theta\,\lambda_{\max}(X) + (1-\theta)\,\lambda_{\max}(Y)
$$

and taking the supremum over $\mathbf{v}$ on the left preserves the bound. (Similarly $\lambda_{\min}$ is concave — an infimum of linear maps.)

$$
\boxed{\mathbb{S}^n_+ \text{ is a convex cone, and } \lambda_{\max}: \mathbb{S}^n \to \mathbb{R} \text{ is convex}}
$$

> **Key takeaway:** Matrix convexity comes cheap once you see spectral quantities as suprema of linear functionals — the same pointwise-supremum theorem that made $\max_i f_i$ convex.

In [20]:
# Problem L3.3 - the PSD cone is convex and lambda_max is a convex function
def sym(n):
    M = rng.normal(size=(n, n))
    return (M + M.T) / 2.0

bad_max = bad_min = 0
for _ in range(2000):
    X, Y, t = sym(4), sym(4), rng.uniform()
    mix = np.linalg.eigvalsh(t * X + (1 - t) * Y)
    if mix.max() > t * np.linalg.eigvalsh(X).max() + (1 - t) * np.linalg.eigvalsh(Y).max() + 1e-12:
        bad_max += 1
    if mix.min() < t * np.linalg.eigvalsh(X).min() + (1 - t) * np.linalg.eigvalsh(Y).min() - 1e-12:
        bad_min += 1
print(f"convexity violations of lambda_max : {bad_max} of 2000")
print(f"concavity violations of lambda_min : {bad_min} of 2000")

Xp, Yp = sym(4), sym(4)
Xp, Yp = Xp @ Xp.T, Yp @ Yp.T
t = 0.3
print(f"PSD cone closed under the combination t X + (1-t) Y : "
      f"{np.linalg.eigvalsh(t * Xp + (1 - t) * Yp).min() >= -1e-12}")
v = rng.normal(size=4)
v = v / np.linalg.norm(v)
print(f"variational bound: v^T X v = {v @ Xp @ v:.6f} <= lambda_max(X) = {np.linalg.eigvalsh(Xp).max():.6f}")
assert bad_max == 0 and bad_min == 0 and v @ Xp @ v <= np.linalg.eigvalsh(Xp).max() + 1e-12

convexity violations of lambda_max : 0 of 2000
concavity violations of lambda_min : 0 of 2000
PSD cone closed under the combination t X + (1-t) Y : True
variational bound: v^T X v = 5.059984 <= lambda_max(X) = 10.598123


### Problem L3.4 — Strong Convexity — Quadratic Lower Bounds and Coercivity

**Problem Statement:** Let $f$ be differentiable and $\mu$-strongly convex on $\mathbb{R}^n$ (i.e. $g(\mathbf{x}) = f(\mathbf{x}) - \frac{\mu}{2}\lVert\mathbf{x}\rVert_2^2$ is convex). Prove: (a) the strong first-order bound

$$
f(\mathbf{y}) \ge f(\mathbf{x}) + \nabla f(\mathbf{x})^\top(\mathbf{y}-\mathbf{x}) + \frac{\mu}{2}\lVert\mathbf{y}-\mathbf{x}\rVert_2^2 \quad \forall\,\mathbf{x},\mathbf{y}
$$

(b) $f$ is coercive; (c) $f$ has exactly one global minimizer $\mathbf{x}^{\star}$, and every $\mathbf{x}$ satisfies $f(\mathbf{x}) - f(\mathbf{x}^{\star}) \ge \frac{\mu}{2}\lVert\mathbf{x}-\mathbf{x}^{\star}\rVert_2^2$.

*Intuition:* Strong convexity plants a parabola under the function at every point; parabolas grow without bound and have unique bottoms, and the function inherits both properties.

**Solution:**

**Step 1 (a).** Apply the ordinary first-order characterization to the convex $g$: for all $\mathbf{x}, \mathbf{y}$,

$$
g(\mathbf{y}) \ge g(\mathbf{x}) + \nabla g(\mathbf{x})^\top(\mathbf{y}-\mathbf{x}), \qquad \nabla g(\mathbf{x}) = \nabla f(\mathbf{x}) - \mu\mathbf{x}
$$

Substituting $g = f - \frac{\mu}{2}\lVert\cdot\rVert_2^2$ and expanding,

$$
f(\mathbf{y}) - \frac{\mu}{2}\lVert\mathbf{y}\rVert_2^2 \ge f(\mathbf{x}) - \frac{\mu}{2}\lVert\mathbf{x}\rVert_2^2 + \nabla f(\mathbf{x})^\top(\mathbf{y}-\mathbf{x}) - \mu\,\mathbf{x}^\top(\mathbf{y}-\mathbf{x})
$$

Now collect the quadratic pieces: $\frac{\mu}{2}\lVert\mathbf{y}\rVert_2^2 - \frac{\mu}{2}\lVert\mathbf{x}\rVert_2^2 - \mu\,\mathbf{x}^\top(\mathbf{y}-\mathbf{x}) = \frac{\mu}{2}\lVert\mathbf{y}-\mathbf{x}\rVert_2^2$ (expand the right side to check). This yields exactly the claimed bound.

**Step 2 (b: coercivity).** Fix $\mathbf{x} = \mathbf{0}$ in (a):

$$
f(\mathbf{y}) \ge f(\mathbf{0}) + \nabla f(\mathbf{0})^\top \mathbf{y} + \frac{\mu}{2}\lVert\mathbf{y}\rVert_2^2 \ge f(\mathbf{0}) - \lVert\nabla f(\mathbf{0})\rVert_2\,\lVert\mathbf{y}\rVert_2 + \frac{\mu}{2}\lVert\mathbf{y}\rVert_2^2
$$

by Cauchy–Schwarz. The right side is a quadratic in $\lVert\mathbf{y}\rVert_2$ with positive leading coefficient, so it tends to $+\infty$ as $\lVert\mathbf{y}\rVert_2 \to \infty$: $f$ is coercive.

**Step 3 (c: existence, uniqueness, growth).** Coercive plus continuous gives a global minimizer $\mathbf{x}^{\star}$: by coercivity some sublevel set $S_\alpha$ is bounded, it is closed by continuity, hence compact, and Weierstrass' theorem gives a minimum on it, which is global because every point outside $S_\alpha$ has a larger value. Strong convexity implies strict convexity, so $\mathbf{x}^{\star}$ is unique. Finally, put $\mathbf{x} = \mathbf{x}^{\star}$ in (a) and use $\nabla f(\mathbf{x}^{\star}) = \mathbf{0}$ (Fermat):

$$
f(\mathbf{y}) \ge f(\mathbf{x}^{\star}) + \frac{\mu}{2}\lVert\mathbf{y}-\mathbf{x}^{\star}\rVert_2^2 \quad \forall\,\mathbf{y}
$$

This quadratic-growth bound converts function-value progress of an algorithm directly into distance-to-solution progress — the key lemma behind linear convergence rates of gradient descent on strongly convex objectives.

$$
\boxed{\mu\text{-strong convexity} \implies \text{coercivity, a unique } \mathbf{x}^{\star}, \text{ and } f(\mathbf{y}) - f(\mathbf{x}^{\star}) \ge \tfrac{\mu}{2}\lVert\mathbf{y}-\mathbf{x}^{\star}\rVert_2^2}
$$

> **Key takeaway:** Strong convexity is the bridge from geometry to algorithms: one inequality simultaneously guarantees that a solution exists, that it is unique, and that optimization error controls parameter error.

In [21]:
# Problem L3.4 - the strong first-order bound, coercivity and quadratic growth
Q = np.array([[3.0, 1.0], [1.0, 2.0]])
c = np.array([1.0, -2.0])
mu = np.linalg.eigvalsh(Q).min()
f_s = lambda z: 0.5 * z @ Q @ z + c @ z
g_s = lambda z: Q @ z + c
x_star = np.linalg.solve(Q, -c)
print(f"mu = lambda_min(Q) = {mu:.6f}")
print(f"minimizer x*       = {x_star}, gradient there = {g_s(x_star)}")

xs = rng.normal(size=(4000, 2)) * 3.0
ys = rng.normal(size=(4000, 2)) * 3.0
slack = np.array([f_s(b) - f_s(a) - g_s(a) @ (b - a) - mu / 2 * np.sum((b - a) ** 2)
                  for a, b in zip(xs, ys)])
growth = np.array([f_s(b) - f_s(x_star) - mu / 2 * np.sum((b - x_star) ** 2) for b in ys])
print(f"min slack in the strong first-order bound : {slack.min():.3e}  (>= 0)")
print(f"min slack in the quadratic-growth bound   : {growth.min():.3e}  (>= 0)")
print(f"coercivity along (1,1) at radius 10, 100, 1000 : "
      f"{np.array([f_s(t * np.array([1.0, 1.0])) for t in (10, 100, 1000)])}")
assert slack.min() >= -1e-9 and growth.min() >= -1e-9 and np.allclose(g_s(x_star), 0.0)

mu = lambda_min(Q) = 1.381966
minimizer x*       = [-0.8  1.4], gradient there = [-0.  0.]


min slack in the strong first-order bound : 6.548e-08  (>= 0)
min slack in the quadratic-growth bound   : 5.589e-07  (>= 0)
coercivity along (1,1) at radius 10, 100, 1000 : [    340.   34900. 3499000.]
